# Baseline Bayesian agent on a Watts–Strogatz network

Agents update beliefs in log-odds space:

$$\ell_i \leftarrow \ell_i + \text{gain} \cdot w_{\text{novelty}} \cdot w_{\text{compat}} \cdot w_{\text{source}} \cdot w_{\text{salience}} \cdot \text{LLR}(c)$$

**Baseline:** all weights = 1 except $w_{\text{novelty}}$ = 0 on repeated claims (normative de-duplication). No biases active.

In [ ]:
from rankers import Config, run, run_replicates, select_neighbor, select_external
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

## 1 — Single run

In [ ]:
cfg = Config(
    n         = 2_000,
    k         = 6,
    p_rewire  = 0.1,
    n_claims  = 200,
    llr_std   = 1.0,
    n_steps   = 2000,
    seed      = 42,
)

result = run(cfg, selection=select_neighbor)
print(f"Elapsed : {result.elapsed_s:.3f} s")
print(f"Final variance   : {result.history['variance'][-1]:.4f}")
print(f"Final bimodality : {result.history['bimodality'][-1]:.4f}  (threshold 0.555)")

In [ ]:
t = np.arange(len(result.history['variance']))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f"Baseline — WS(N={cfg.n}, k={cfg.k}, p={cfg.p_rewire})  |  single run", y=1.01)

axes[0].plot(t, result.history['variance'], color='steelblue')
axes[0].set(title='Belief variance', xlabel='Step', ylabel='Var($\\ell$)')

axes[1].plot(t, result.history['bimodality'], color='darkorange')
axes[1].axhline(5/9, ls='--', color='gray', lw=1, label='threshold 5/9')
axes[1].set(title='Bimodality coefficient $B$', xlabel='Step')
axes[1].legend(fontsize=8)

axes[2].hist(result.final_beliefs, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[2].axvline(0, ls='--', color='red', alpha=0.7, lw=1)
axes[2].set(title='Final belief distribution', xlabel='Log-odds $\\ell$', ylabel='count')

plt.tight_layout()
plt.show()

## 2 — Network vs. external selection

**`select_neighbor`**: each agent receives the last-broadcast claim of a random neighbour — network topology shapes exposure.  
**`select_external`**: each agent draws independently from the full claim pool — no network effect.  

Comparing the two isolates the structural contribution of the WS graph.

In [ ]:
N_REPS = 20

agg_net = run_replicates(cfg, n_reps=N_REPS, selection=select_neighbor)
agg_ext = run_replicates(cfg, n_reps=N_REPS, selection=select_external)

print(f"Network  final variance: {agg_net['mean']['variance'][-1]:.4f}")
print(f"External final variance: {agg_ext['mean']['variance'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"Network vs. external selection  —  {N_REPS} replicates each", y=1.01)

for ax, key, ylabel in zip(
    axes,
    ['variance', 'bimodality'],
    ['Var($\\ell$)', 'Bimodality $B$'],
):
    for agg, label, color in [
        (agg_net, 'network',  'steelblue'),
        (agg_ext, 'external', 'tomato'),
    ]:
        mu  = agg['mean'][key]
        sd  = agg['std'][key]
        tt  = np.arange(len(mu))
        ax.plot(tt, mu, color=color, label=label)
        ax.fill_between(tt, mu - sd, mu + sd, color=color, alpha=0.15)

    if key == 'bimodality':
        ax.axhline(5/9, ls='--', color='gray', lw=1, label='threshold')

    ax.set(title=ylabel, xlabel='Step', ylabel=ylabel)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3 — Final belief distributions across replicates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
fig.suptitle('Final belief distributions across replicates', y=1.01)

for ax, agg, title, color in [
    (axes[0], agg_net, 'Network selection',  'steelblue'),
    (axes[1], agg_ext, 'External selection', 'tomato'),
]:
    beliefs = agg['final_beliefs'].ravel()   # (n_reps * N,)
    ax.hist(beliefs, bins=80, color=color, edgecolor='white', linewidth=0.2, density=True)
    ax.axvline(0, ls='--', color='black', alpha=0.5, lw=1)
    ax.set(title=title, xlabel='Log-odds $\\ell$', ylabel='density')

plt.tight_layout()
plt.show()

## 4 — Sensitivity: rewiring probability

How does WS topology (from ring lattice at $p=0$ to random graph at $p=1$) affect polarisation?

In [ ]:
p_values = [0.0, 0.01, 0.05, 0.1, 0.3, 1.0]
N_REPS_SENS = 10

final_var  = []
final_bimo = []

for p in p_values:
    agg = run_replicates(
        Config(**{**cfg.__dict__, 'p_rewire': p, 'seed': 0}),
        n_reps=N_REPS_SENS,
        selection=select_neighbor,
    )
    final_var.append(agg['mean']['variance'][-1])
    final_bimo.append(agg['mean']['bimodality'][-1])
    print(f"p={p:.2f}  var={final_var[-1]:.3f}  B={final_bimo[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Sensitivity to WS rewiring probability $p$', y=1.01)

axes[0].plot(p_values, final_var, 'o-', color='steelblue')
axes[0].set(title='Final variance', xlabel='Rewiring probability $p$', ylabel='Var($\\ell$)')

axes[1].plot(p_values, final_bimo, 'o-', color='darkorange')
axes[1].axhline(5/9, ls='--', color='gray', lw=1, label='threshold')
axes[1].set(title='Final bimodality $B$', xlabel='Rewiring probability $p$')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()